# Virtual Drum Kit

본 프로젝트는 Jupyter Notebook 환경에서 실행 가능한 가상 드럼 연주 프로그램이다.  
사용자는 마우스 클릭 또는 키보드 입력을 통해 다양한 드럼 사운드를 재생할 수 있다.

구현한 드럼 사운드는 Bell, Crash, High Tom, Middle Tom, Low Tom, Kick, Ride, Snare, Hi-hat 총 9가지이다.  
각 드럼 패드는 개별 오디오 파일과 연결되어 있으며, 사용자의 입력이 발생하면 해당 사운드가 즉시 재생된다. 또한 입력 시 패드가 강조되는 시각적 피드백을 제공하여 실제 악기를 연주하는 듯한 상호작용을 구현하였다.

## 개발 목적

본 프로그램의 목적은 사용자가 별도의 악기 없이도 간단한 드럼 연주를 경험할 수 있도록 하는 것이다.  
드럼 연주 프로그램은 사용자 입력 처리, 오디오 재생, 이벤트 기반 프로그래밍, 시각적 피드백을 포함하기 때문에 기초 프로그래밍 개념을 종합적으로 적용하기에 적합하다.

또한 단순한 버튼 클릭 프로그램에서 확장하여, 실제 드럼셋의 구성 요소를 반영한 인터페이스를 설계함으로써 사용자가 직관적으로 악기를 연주할 수 있도록 하였다.

## 주요 기능

1. Bell, Crash, High Tom, Middle Tom, Low Tom, Kick, Ride, Snare, Hi-hat 총 9개의 드럼 패드 제공
2. 마우스 클릭을 통한 드럼 사운드 재생
3. 키보드 입력을 통한 드럼 사운드 재생
4. 입력 시 드럼 패드 강조 애니메이션 제공
5. 오디오 파일을 base64 형식으로 변환하여 Jupyter Notebook 내부에서 안정적으로 재생

## 조작 방법
- **마우스 클릭**: 각 패드를 클릭하면 해당 사운드가 재생됩니다.
- **키보드 입력**: 아래 키를 누르면 해당 사운드가 재생됩니다.

| 키 | 사운드 | 파일 |
|---|---|---|
| Q | Bell | `sounds/bell.wav` |
| W | Crash | `sounds/crash.wav` |
| E | Ride | `sounds/ride.wav` |
| A | Hi-hat | `sounds/hihat.wav` |
| S | Snare | `sounds/snare.wav` |
| D | Kick | `sounds/kick.wav` |
| Z | High Tom | `sounds/htom.wav` |
| X | Middle Tom | `sounds/mtom.wav` |
| C | Low Tom | `sounds/ltom.wav` |

> 브라우저 정책에 따라, **처음에는 한 번 클릭**이 있어야 키보드로도 소리가 나는 경우가 있습니다.

In [13]:
from pathlib import Path
import uuid
import base64
import webbrowser
from urllib.request import pathname2url
from IPython.display import display, HTML

# (1) 사운드 파일 경로(기존 구조 유지)
sound_files = {
    "bell": "sounds/bell.wav",
    "crash": "sounds/crash.wav",
    "ride": "sounds/ride.wav",
    "hihat": "sounds/hihat.wav",
    "snare": "sounds/snare.wav",
    "kick": "sounds/kick.wav",
    "htom": "sounds/htom.wav",
    "mtom": "sounds/mtom.wav",
    "ltom": "sounds/ltom.wav",
}

# (2) 파일 존재 확인(없으면 기존처럼 예외)
missing = [p for p in sound_files.values() if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing sound file(s): " + ", ".join(missing))

# (3) wav → base64 data URI 변환
def wav_to_data_uri(path: str) -> str:
    raw = Path(path).read_bytes()
    b64 = base64.b64encode(raw).decode("ascii")
    return f"data:audio/wav;base64,{b64}"

sound_data = {name: wav_to_data_uri(path) for name, path in sound_files.items()}

# (4) pads 구조 유지, sound만 base64 data URI로
pads = [
    {"key": "KeyQ", "label": "Bell",       "sound": sound_data["bell"]},
    {"key": "KeyW", "label": "Crash",      "sound": sound_data["crash"]},
    {"key": "KeyE", "label": "Ride",       "sound": sound_data["ride"]},
    {"key": "KeyA", "label": "Hi-hat",     "sound": sound_data["hihat"]},
    {"key": "KeyS", "label": "Snare",      "sound": sound_data["snare"]},
    {"key": "KeyD", "label": "Kick",       "sound": sound_data["kick"]},
    {"key": "KeyZ", "label": "High Tom",   "sound": sound_data["htom"]},
    {"key": "KeyX", "label": "Middle Tom", "sound": sound_data["mtom"]},
    {"key": "KeyC", "label": "Low Tom",    "sound": sound_data["ltom"]},
]

root_id = f"drum-kit-{uuid.uuid4().hex}"

# 배치/스타일 용도(기능 로직은 동일)
label_to_slot = {
    "Crash": "crash",
    "Ride": "ride",
    "Bell": "bell",
    "Hi-hat": "hihat",
    "High Tom": "htom",
    "Middle Tom": "mtom",
    "Snare": "snare",
    "Low Tom": "ltom",
    "Kick": "kick",
}
cymbals = {"Crash", "Ride", "Bell", "Hi-hat"}

tiles_html = "\n".join(
    [
        (
            f"""
<button class=\"drum-pad pad-{label_to_slot[p['label']]} {'cymbal' if p['label'] in cymbals else ''} {'kick' if p['label'] == 'Kick' else ''} {'snare' if p['label'] == 'Snare' else ''}\" type=\"button\" data-key=\"{p['key']}\" data-sound=\"{p['sound']}\" aria-label=\"{p['label']} ({p['key'].replace('Key','')})\">
  <div class=\"drum-label\">{p['label']}</div>
  <div class=\"drum-key\">{p['key'].replace('Key', '')}</div>
</button>
""".strip()
        )
        for p in pads
    ]
)

# (5) 단독 실행 가능한 HTML 문서 생성
html = f"""<!doctype html>
<html lang=\"ko\">
<head>
  <meta charset=\"utf-8\" />
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
  <title>Virtual Drum Kit</title>
</head>
<body>
  <div class=\"page\">
    <h1 class=\"title\">Virtual Drum Kit</h1>
    <div class=\"subtitle\">클릭 또는 Q/W/E/A/S/D/Z/X/C 키로 연주하세요.</div>
    <div class=\"note\">브라우저 정책에 따라 처음에는 한 번 클릭이 필요할 수 있습니다.</div>
    <div class=\"drum-kit\" id=\"{root_id}\" role=\"application\" aria-label=\"Virtual Drum Kit\">
      <div class=\"drum-stage\">
        {tiles_html}
      </div>
    </div>
  </div>

<style>
  body {{ margin: 0; padding: 0; font-family: system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial, 'Apple SD Gothic Neo', 'Noto Sans KR', sans-serif; }}
  .page {{ max-width: 980px; margin: 0 auto; padding: 18px 16px 28px; }}
  .title {{ margin: 0 0 6px 0; font-size: 22px; }}
  .subtitle {{ font-size: 13px; opacity: 0.75; margin-bottom: 6px; }}
  .note {{ font-size: 12px; opacity: 0.65; margin-bottom: 12px; }}

  .drum-kit {{ max-width: 860px; }}
  .drum-stage {{
    position: relative;
    width: 100%;
    height: 520px;
    border-radius: 18px;
    border: 1px solid rgba(0, 0, 0, 0.08);
    background: rgba(0, 0, 0, 0.02);
    overflow: hidden;
  }}

  .drum-pad {{
    position: absolute;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    gap: 6px;
    user-select: none;
    border: 2px solid rgba(0, 0, 0, 0.18);
    border-radius: 9999px;
    padding: 10px;
    background: rgba(0, 0, 0, 0.04);
    cursor: pointer;
    text-align: center;
    --base-transform: none;
    transform: var(--base-transform);
    transition: transform 90ms ease, background 140ms ease, border-color 140ms ease, filter 140ms ease;
  }}
  .drum-pad:active {{
    transform: var(--base-transform) scale(0.98);
  }}
  .drum-pad.active {{
    transform: var(--base-transform) scale(1.04);
    border-color: rgba(0, 0, 0, 0.35);
    background: rgba(0, 0, 0, 0.08);
    filter: brightness(1.05);
  }}

  .drum-pad.cymbal {{
    background: rgba(245, 205, 70, 0.65);
    border-color: rgba(160, 120, 0, 0.25);
  }}
  .drum-pad.cymbal.active {{
    background: rgba(245, 205, 70, 0.78);
    border-color: rgba(160, 120, 0, 0.45);
    filter: brightness(1.08);
  }}
  .pad-bell.cymbal {{
    background: rgba(245, 205, 70, 0.82);
    border-color: rgba(160, 120, 0, 0.45);
  }}

  .drum-pad.snare {{
    background: rgba(120, 170, 255, 0.35);
    border-color: rgba(40, 80, 160, 0.25);
  }}
  .drum-pad.snare.active {{
    background: rgba(120, 170, 255, 0.5);
    border-color: rgba(40, 80, 160, 0.45);
    filter: brightness(1.06);
  }}

  .drum-label {{ font-size: 14px; font-weight: 700; line-height: 1.1; }}
  .drum-key {{
    font-size: 12px;
    padding: 4px 10px;
    border-radius: 999px;
    border: 1px solid rgba(0, 0, 0, 0.22);
    background: rgba(255, 255, 255, 0.75);
  }}

  /* 드럼셋 배치 */
  .pad-crash {{ width: 150px; height: 150px; left: 5%;  top: 7%;  }}
  .pad-ride  {{ width: 180px; height: 180px; left: 77%; top: 16%; --base-transform: translate(-50%, -50%); z-index: 1; }}
  .pad-bell  {{ width: 110px; height: 110px; left: 77%; top: 16%; --base-transform: translate(-50%, -50%); z-index: 2; }}
  .pad-hihat {{ width: 135px; height: 135px; left: 8%;  top: 38%; }}
  .pad-htom  {{ width: 135px; height: 135px; left: calc(42% + 40px); top: 20%; --base-transform: translateX(-100%); }}
  .pad-mtom  {{ width: 135px; height: 135px; left: 49%; top: 20%; }}
  .pad-snare {{ width: 150px; height: 150px; left: 28%; top: 52%; z-index: 3; }}
  .pad-ltom  {{ width: 160px; height: 160px; right: calc(14% - 40px); top: 58%; }}
  .pad-kick  {{ width: 230px; height: 230px; left: 50%; bottom: 6%; --base-transform: translateX(-50%); z-index: 0; }}
</style>

<script>
(() => {{
  const root = document.getElementById('{root_id}');
  if (!root) return;

  const pads = Array.from(root.querySelectorAll('.drum-pad'));
  const audioByKey = new Map();

  function getAudio(key, soundPath) {{
    if (audioByKey.has(key)) return audioByKey.get(key);
    const audio = new Audio(soundPath);
    audio.preload = 'auto';
    audioByKey.set(key, audio);
    return audio;
  }}

  function flash(el) {{
    el.classList.add('active');
    window.setTimeout(() => el.classList.remove('active'), 120);
  }}

  async function playForPad(padEl) {{
    const key = padEl.dataset.key;
    const sound = padEl.dataset.sound;
    const audio = getAudio(key, sound);
    try {{
      audio.currentTime = 0;
      const p = audio.play();
      if (p && typeof p.then === 'function') await p;
    }} catch (e) {{
      console.warn('Audio play blocked:', e);
    }}
    flash(padEl);
  }}

  pads.forEach(padEl => {{
    padEl.addEventListener('click', () => playForPad(padEl));
  }});

  window.addEventListener('keydown', (ev) => {{
    const code = ev.code;
    const target = pads.find(p => p.dataset.key === code);
    if (!target) return;
    if (ev.repeat) return;
    playForPad(target);
  }});
}})();
</script>
</body>
</html>
"""

# (6) 파일 생성 + 브라우저로 열기
html_path = Path.cwd() / "drum_kit.html"
html_path.write_text(html, encoding="utf-8")
file_url = "file:" + pathname2url(str(html_path.resolve()))

opened = False
try:
    opened = webbrowser.open(file_url, new=1, autoraise=True)
except Exception as e:
    print("webbrowser.open() failed:", e)

# 노트북에는 링크만 표시(실제 실행은 브라우저에서)
display(HTML(f"""<div>✅ <b>drum_kit.html</b> 생성: <a href='{file_url}' target='_blank'>{html_path.name}</a><br/>브라우저 열기 시도: {opened}</div>"""))